In [1]:
from pathlib import Path
import pandas as pd

project_root = Path.cwd()

if project_root.name == "notebooks":
    project_root = project_root.parent

project_root

WindowsPath('c:/Users/Dell - i5 11th Gen/Desktop/clinvar-gnomad-vus-analyzer')

In [2]:
candidate_file = (
    project_root
    / "data"
    / "processed"
    / "atm_clinvar_candidates.csv"
)

candidate_variants = pd.read_csv(candidate_file)

candidate_variants.head()

,Variation,Genes,Type,Condition,Classification,Review status,Molecular consequence,Protein change,Germline date last evaluated,Somatic clinical impact date last evaluated,Oncogenicity date last evaluated,GRCh37 Location,GRCh38 Location,Variation ID,VCV accession,rs_ID
0,NM_000051.4(ATM):c.4A>C (p.Ser2Arg),ATM,single nucleotide variant,Hereditary cancer-predisposing syndrome,G: Uncertain significance,"criteria provided, single submitter",missense variant,S2R,2025/08/26,NaN,NaN,11:108098355,11:108227628,4170131,VCV004170131,rs4279232
1,NM_000051.4(ATM):c.4A>G (p.Ser2Gly),ATM,single nucleotide variant,"Hereditary cancer-predisposing syndrome, Famil...",G: Uncertain significance,"criteria provided, multiple submitters, no con...",missense variant,S2G,2024/02/19,NaN,NaN,11:108098355,11:108227628,640846,VCV000640846,rs639367
2,NM_000051.4(ATM):c.5G>C (p.Ser2Thr),ATM,single nucleotide variant,not specified,G: Uncertain significance,"criteria provided, single submitter",missense variant,S2T,2014/06/23,NaN,NaN,11:108098356,11:108227629,181943,VCV000181943,rs180377
3,NM_000051.4(ATM):c.6T>G (p.Ser2Arg),ATM,single nucleotide variant,not specified,G: Uncertain significance,"criteria provided, single submitter",missense variant,S2R,2019/12/09,NaN,NaN,11:108098357,11:108227630,1337452,VCV001337452,rs1328461
4,NM_000051.4(ATM):c.6T>A (p.Ser2Arg),ATM,single nucleotide variant,"Hereditary cancer-predisposing syndrome, Ataxi...",G: Uncertain significance,"criteria provided, multiple submitters, no con...",missense variant,S2R,2025/11/01,NaN,NaN,11:108098357,11:108227630,922075,VCV000922075,rs911284


In [3]:
import gzip

vcf_file = (
    project_root
    / "data"
    / "raw"
    / "clinvar_grch38.vcf.gz"
)

print(vcf_file.exists())

# why gzip and rt ??
with gzip.open(vcf_file, "rt") as file:
    for line in file:
        if not line.startswith("##"):
            print(line.strip())
            break

True
#CHROM	POS	ID	REF	ALT	QUAL	FILTER	INFO


In [4]:
vcf_columns = [
    "CHROM",
    "POS",
    "ID",
    "REF",
    "ALT",
    "QUAL",
    "FILTER",
    "INFO"
]

candidate_ids = set(
    pd.to_numeric(
        candidate_variants["Variation ID"],
        errors="coerce"
    )
    .dropna()
    .astype("int64")
    .astype(str)
)

print("Candidate IDs:", len(candidate_ids))

Candidate IDs: 4534


In [5]:
matching_chunks = []

for chunk in pd.read_csv(
    vcf_file,
    sep="\t",
    comment="#",
    names=vcf_columns,
    dtype={
        "CHROM": "string",
        "ID": "string",
        "REF": "string",
        "ALT": "string"
    },
    chunksize=100_000,
    low_memory=False
):
    matches = chunk[
        chunk["ID"].isin(candidate_ids)
    ]

    if not matches.empty:
        matching_chunks.append(matches)
        

In [6]:
if not matching_chunks:
    raise ValueError("No matching ClinVar IDs were found in the VCF.")

clinvar_vcf_matches = pd.concat(
    matching_chunks,
    ignore_index=True
)

print("VCF matches found:", len(clinvar_vcf_matches))

clinvar_vcf_matches.head()

VCF matches found: 4534


,CHROM,POS,ID,REF,ALT,QUAL,FILTER,INFO
0,11,108227628,4170131,A,C,.,.,"ALLELEID=4279232;CLNDISDB=MONDO:MONDO:0015356,..."
1,11,108227628,640846,A,G,.,.,"ALLELEID=639367;CLNDISDB=MONDO:MONDO:0015356,M..."
2,11,108227629,181943,G,C,.,.,ALLELEID=180377;CLNDISDB=MedGen:CN169374;CLNDN...
3,11,108227630,922075,T,A,.,.,"ALLELEID=911284;CLNDISDB=MONDO:MONDO:0015356,M..."
4,11,108227630,1337452,T,G,.,.,AF_EXAC=0.00001;ALLELEID=1328461;CLNDISDB=MedG...


In [7]:
clinvar_vcf_matches["Variation ID"] = (
    pd.to_numeric(
        clinvar_vcf_matches["ID"],
        errors="coerce"
    )
    .astype("Int64")
)

clinvar_vcf_matches["variant_id"] = (
    clinvar_vcf_matches["CHROM"].astype(str)
    + "-"
    + clinvar_vcf_matches["POS"].astype(str)
    + "-"
    + clinvar_vcf_matches["REF"].astype(str)
    + "-"
    + clinvar_vcf_matches["ALT"].astype(str)
)

clinvar_vcf_matches[
    [
        "Variation ID",
        "CHROM",
        "POS",
        "REF",
        "ALT",
        "variant_id"
    ]
].head()

,Variation ID,CHROM,POS,REF,ALT,variant_id
0,4170131,11,108227628,A,C,11-108227628-A-C
1,640846,11,108227628,A,G,11-108227628-A-G
2,181943,11,108227629,G,C,11-108227629-G-C
3,922075,11,108227630,T,A,11-108227630-T-A
4,1337452,11,108227630,T,G,11-108227630-T-G


In [8]:
candidates_with_variant_id = candidate_variants.merge(
    clinvar_vcf_matches[
        [
        "Variation ID", 
        "CHROM",
        "POS",
        "REF",
        "ALT",
        "variant_id"]
    ]
    .rename(
        columns={
            "CHROM": "chromosome",
            "POS": "position",
            "REF": "reference_allele",
            "ALT": "alternate_allele"
        }),
    on="Variation ID",
    how="inner"
)

print("Candidates with variant IDs:", len(candidates_with_variant_id))

candidates_with_variant_id.head()

Candidates with variant IDs: 4534


,Variation,Genes,Type,Condition,Classification,Review status,Molecular consequence,Protein change,Germline date last evaluated,Somatic clinical impact date last evaluated,...,GRCh37 Location,GRCh38 Location,Variation ID,VCV accession,rs_ID,chromosome,position,reference_allele,alternate_allele,variant_id
0,NM_000051.4(ATM):c.4A>C (p.Ser2Arg),ATM,single nucleotide variant,Hereditary cancer-predisposing syndrome,G: Uncertain significance,"criteria provided, single submitter",missense variant,S2R,2025/08/26,NaN,...,11:108098355,11:108227628,4170131,VCV004170131,rs4279232,11,108227628,A,C,11-108227628-A-C
1,NM_000051.4(ATM):c.4A>G (p.Ser2Gly),ATM,single nucleotide variant,"Hereditary cancer-predisposing syndrome, Famil...",G: Uncertain significance,"criteria provided, multiple submitters, no con...",missense variant,S2G,2024/02/19,NaN,...,11:108098355,11:108227628,640846,VCV000640846,rs639367,11,108227628,A,G,11-108227628-A-G
2,NM_000051.4(ATM):c.5G>C (p.Ser2Thr),ATM,single nucleotide variant,not specified,G: Uncertain significance,"criteria provided, single submitter",missense variant,S2T,2014/06/23,NaN,...,11:108098356,11:108227629,181943,VCV000181943,rs180377,11,108227629,G,C,11-108227629-G-C
3,NM_000051.4(ATM):c.6T>G (p.Ser2Arg),ATM,single nucleotide variant,not specified,G: Uncertain significance,"criteria provided, single submitter",missense variant,S2R,2019/12/09,NaN,...,11:108098357,11:108227630,1337452,VCV001337452,rs1328461,11,108227630,T,G,11-108227630-T-G
4,NM_000051.4(ATM):c.6T>A (p.Ser2Arg),ATM,single nucleotide variant,"Hereditary cancer-predisposing syndrome, Ataxi...",G: Uncertain significance,"criteria provided, multiple submitters, no con...",missense variant,S2R,2025/11/01,NaN,...,11:108098357,11:108227630,922075,VCV000922075,rs911284,11,108227630,T,A,11-108227630-T-A


In [9]:
output_file = (
    project_root
    / "data"
    / "processed"
    / "atm_candidates_with_variant_id.csv"
)

candidates_with_variant_id.to_csv(
    output_file,
    index=False
)

print("Saved to:", output_file)

Saved to: c:\Users\Dell - i5 11th Gen\Desktop\clinvar-gnomad-vus-analyzer\data\processed\atm_candidates_with_variant_id.csv
